# FULL-7 Final Clean Package — zweites Google-Konto\n\nDieses Notebook ist ausschließlich für **ecomercetatek@gmail.com** vorgesehen. Es lädt die bereits verifizierten, freigegebenen 14 Recovery-Teile plus `FootyStats_TopLigen_2026(1).zip` aus den mit diesem Konto geteilten Dateien, prüft Größe und SHA-256, und erstellt in **My Drive** genau eine ZIP mit exakt 15 Einträgen.\n\nEs führt **keine Collection, kein Training und keine API-Nachsammlung** durch. Die Recovery-Key-Datei wird bewusst **nicht** in die 15-Dateien-ZIP aufgenommen.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
from pathlib import Path
import zipfile, hashlib, json, os, shutil, io

EXPECTED_ACCOUNT = "ecomercetatek@gmail.com"
OUT_DIR = Path("/content/drive/MyDrive/FULL7_FINAL_CLEAN_PACKAGE_2026-09-23")
OUT = OUT_DIR / "FULL7_MASSENSAMMLUNG_PLUS_50_LIGEN_BACKTEST_2026-09-23.zip"
TMP = OUT_DIR / (OUT.name + ".partial")
REPORT = OUT_DIR / "FULL7_MASSENSAMMLUNG_PLUS_50_LIGEN_BACKTEST_2026-09-23_VERIFICATION.json"
LOCAL = Path("/content/FULL7_FINAL_SOURCE")

FILES = [
    ("part-0000.tar.enc","1_Yh4nQZiZ5-1tFX1EnFrqcDnldGRUHQK",798996512,"cb0eb0f2158963e0c0b336718b88e0ad611dc8025ccfdba40b52d5a3e31a1fac"),
    ("part-0001.tar.enc","1bAlClktyr2Ql7P_GFE32mVNGUlD54BDz",799488032,"9b149fafaa6f698fe04d44caff01c7dea5b73c30eb77d2b64fca8e3a0f6bf017"),
    ("part-0002.tar.enc","1MVRtf8OlmCpWjVAG0QyOdc6hXMMbFZB0",799610912,"7dbc0c571de453db994e2f804fdfc96b0054f5f5f61bbb8563351cff21c36bb4"),
    ("part-0003.tar.enc","1dn7qj5fQiK5fOJZXN-fazTAEkcKe-nxe",799180832,"abfc43fe2d7268997b678f120376d470bcb06e93bf93d2dcaff081fcf8bc1338"),
    ("part-0004.tar.enc","1pWMTmDJZiOmcSxABu2oGahqhJ1kyrmkV",799303712,"f29640681bb29e25e8dae3628948cecfb4d32fbf8852e34114c5638690c078d9"),
    ("part-0005.tar.enc","14On_7t33GVvp-rc9vJ2_GqvjIAOoSS9C",799539232,"25a9a306e603aa69935b64ffcbffc39fedaa09aba1f4d5593e507ceb76853326"),
    ("part-0006.tar.enc","1F775fjpFXjN9ciPofirv8-HUqFtv6XMp",799406112,"893d15b8345bbd0057e123c59ea7004e1d8db177bcd5188ffeccb079ca285543"),
    ("part-0007.tar.enc","1ZalzABSOTS5j06G4ZwX2wlVacaQ8_2Ul",800808992,"77d5be4d77fd46bb5ec0fdcfd6268b4686ebcdc3a28779746b9b8e553bb97e90"),
    ("part-0008.tar.enc","1W2c8oPl0ag8GFE3YRear2vcQg6r9M7Wf",799477792,"14167a29596224048c678837ab43e883b972df2ce99446de7ed5b411d8631293"),
    ("part-0009.tar.enc","1KlNUfVICBRSlJQcgMni709U_qZ6vI9O1",799068192,"ad33a6b73d1a65b0b01b860aa0b7d3b7de3e9c7d28b77fcf31d676e7210ef161"),
    ("part-0010.tar.enc","1kC92j6NDtvFFLkUTcYBpqxfpbvKZPqD-",799641632,"1023122f652c26f369dbc61aa9d25a64f0338bb5510bc8d2e0d4ef9aee640b27"),
    ("part-0011.tar.enc","1NedzbaeiySkrUR77Of-uGqCDxz3j-JWt",799569952,"f2a676655da86e902b3f3b424cc72d064b52c7073bc86f3ec2ed814bbecd3409"),
    ("part-0012.tar.enc","14avYOmZHq3vxJvsMaDfT303S7lVjLYI2",799447072,"3bfb97082ec063cd4699063a835092f4e183b8ea4fe10fb5827c3bf0979706f0"),
    ("part-0013.tar.enc","1YeR4WKKLyUcybb7uXT64Jrh6qoRcFCRb",797440032,"d79faa151b831f10120aa378f789861c54132db6c56d73da89f22eef2e1b2fce"),
    ("FootyStats_TopLigen_2026(1).zip","1keW9ryf7VlBi9VyqiuW28TQ7tEaWkb7N",3095290,"ae1510742e576e3c3fb40382995916fc6b4227c11f4b2f365022efda88612298"),
]

creds, _ = google.auth.default()
svc = build("drive", "v3", credentials=creds, cache_discovery=False)

about = svc.about().get(fields="user(emailAddress,displayName),storageQuota(limit,usage,usageInDrive,usageInDriveTrash)").execute()
email = about.get("user", {}).get("emailAddress")
print("AUTHENTICATED ACCOUNT =", email)
if email != EXPECTED_ACCOUNT:
    raise RuntimeError(f"STOP: falsches Google-Konto. Erwartet {EXPECTED_ACCOUNT}, aktiv ist {email}")

quota = about.get("storageQuota", {})
limit = int(quota.get("limit") or 0)
usage = int(quota.get("usage") or 0)
available = limit - usage if limit else None
source_total = sum(x[2] for x in FILES)
needed = source_total + 100_000_000
print(f"SOURCE TOTAL = {source_total:,} bytes")
if available is not None:
    print(f"DRIVE FREE = {available:,} bytes")
    if available < needed:
        raise RuntimeError(f"STOP: zu wenig Platz im zweiten Konto. Frei={available:,}, benötigt mindestens={needed:,}")

LOCAL.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, block=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

def valid_local(path, size, sha):
    return path.is_file() and path.stat().st_size == size and sha256_file(path) == sha

print("\n=== DOWNLOAD SHARED SOURCES TO COLAB LOCAL DISK ===")
for idx, (name, file_id, size, sha) in enumerate(FILES, 1):
    dst = LOCAL / name
    if valid_local(dst, size, sha):
        print(f"[{idx}/15] REUSE verified {name}")
        continue
    if dst.exists():
        dst.unlink()
    req = svc.files().get_media(fileId=file_id)
    with open(dst, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print(f"[{idx}/15] {name}: {status.progress()*100:.1f}%", flush=True)
    if dst.stat().st_size != size:
        raise RuntimeError(f"Size mismatch {name}: {dst.stat().st_size} != {size}")
    got = sha256_file(dst)
    if got != sha:
        raise RuntimeError(f"SHA mismatch {name}: {got} != {sha}")
    print(f"[{idx}/15] VERIFIED {name}")

EXPECTED_NAMES = [x[0] for x in FILES]

if TMP.exists():
    TMP.unlink()

print("\n=== BUILD FINAL CLEAN ZIP: EXACTLY 15 ENTRIES ===")
with zipfile.ZipFile(TMP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for idx, (name, _, _, _) in enumerate(FILES, 1):
        src = LOCAL / name
        print(f"[{idx}/15] adding {name}", flush=True)
        zf.write(src, arcname=name)

if OUT.exists():
    OUT.unlink()
TMP.rename(OUT)

print("\n=== VERIFY ZIP STRUCTURE / CRC ===")
with zipfile.ZipFile(OUT, "r") as zf:
    infos = zf.infolist()
    names = [x.filename for x in infos]
    if len(names) != 15:
        raise RuntimeError(f"Expected 15 entries, got {len(names)}")
    if names != EXPECTED_NAMES:
        raise RuntimeError(f"Entry mismatch. Actual={names}")
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"CRC failure: {bad}")
    sizes = {x.filename: x.file_size for x in infos}
    for name, _, size, _ in FILES:
        if sizes[name] != size:
            raise RuntimeError(f"ZIP size mismatch for {name}")

print("15/15 entries: PASS")
print("CRC: PASS")
print("extra entries: 0")

print("\n=== FINAL OUTER ZIP SHA-256 ===")
zip_sha = sha256_file(OUT)
report = {
    "status": "PASS",
    "authenticated_account": email,
    "package": OUT.name,
    "entry_count": 15,
    "entries": EXPECTED_NAMES,
    "source_total_bytes": source_total,
    "zip_bytes": OUT.stat().st_size,
    "zip_sha256": zip_sha,
    "crc_test": "PASS",
    "extra_entries": 0,
    "contains_only": {
        "full7_recovery_parts": 14,
        "footystats_50_league_backtest": 1,
        "grok": 0,
        "checkpoints": 0,
        "143_333_443_packages": 0,
        "decision_engines": 0,
        "handoffs": 0,
        "metadata_inside_zip": 0
    }
}
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("\nFINAL CLEAN PACKAGE VERIFIED")
print("package =", OUT)
print("entry_count = 15")
print("zip_bytes =", OUT.stat().st_size)
print("zip_sha256 =", zip_sha)
print("verification_report =", REPORT)
